In [ ]:
using ProgressBars, Bessels
include("/Users/travail/Desktop/Quantic/Dev/Stage_quantic_2025/Code_Julia/Julia_file/classical_oscillators.jl");

# Emission spectrum for the two-photon process

In [ ]:
const ∞ = Inf
ħ=1
e=1
β = ∞ # Case zero temperature T=0, β → ∞
ϕ_quantum=1/2

#### Caracterization of the circuit #####

ω_a = 3.7*2*pi
ω_b_0 = 7.05*2*pi

Lamb_shift = - 0.05 #Lamb shift in GHz

ω_b = ω_b_0 #+ Lamb_shift*2*pi # Shifted frequency of mode b

φ_a = 0.11
Z_a = 2*ϕ_quantum^2*φ_a^2
C_a_inv = Z_a*ω_a
E_Ca = C_a_inv/2

φ_b = 0.2
Z_b = 2*ϕ_quantum^2*φ_b^2
C_b_inv = Z_b*ω_b
E_Cb = C_b_inv/2

E_J = 37*2*pi




κ=1/10; # 1/ns from Jeffrey et al.
ϕ_quantum=1/2

ω_d=ω_b_0 # Drive frequency
ω_p=2ω_a-ω_d # Pump frequency

ϵ_0 = 0.25
g2_max = besselj(1,ϵ_0)*E_J*φ_a^2*φ_b # (Leon's paper)
ϵ_d=10*g2_max

κ_2 = 4*g2_max^2/κ;

In [ ]:
n=7000
t_max=15/κ

bath=Exp_decay
bath_params = bath_params=(ω_0=50*2*pi, ω_cut_off=30*2*pi)

AB, J = bath(bath_params=bath_params,κ=κ, C_inv=C_b_inv)
chain_params = chaincoeffs_finiteT(n, β, false; J=J, mc=size(AB)[1], mp=0, AB=AB, iq=1, idelta=2, procedure=:Lanczos, Mmax=7000, save=false)  # chain parameters, i.e. on-site energies ϵ_i, hopping energies t_i, and system-chain coupling c_0

In [ ]:
gq=vcat(- chain_params[2])
gϕ=vcat(chain_params[2])

Mq_chain=SymTridiagonal(chain_params[1], gq)
Mϕ_chain=SymTridiagonal(chain_params[1], gϕ);

In [ ]:
A_chain = Mq_chain * Mϕ_chain

vals_chain, vecs_chain = eigen(A_chain)
ωk²_chain = real(vals_chain)
ωk_chain = sqrt.(ωk²_chain)
P_chain = real(vecs_chain)


Dq_chain = diag(P_chain'*(Mq_chain \ P_chain))  # uses linear solve instead of inv
Dϕ_chain = diag(P_chain' * Mϕ_chain * P_chain);

In [ ]:
ϕ_a,ϕ_b,q_a,q_b,t_list = two_photon_1d_chain(n, t_max;chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b_0,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 =0,κ=κ, ϵ_d=0, ω_d=0),
     init_mode_a=(ϕ_a_0=1.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=1.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=false)
F,freqs=FFT_perso(ϕ_b, t_list)
f_shifted=abs(freqs[argmax(abs.(F))]);

In [ ]:
ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=true)

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ
S_dot = P_chain \ ϕ_dot

# Energy in normal modes
Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
f_a=ω_a/2pi
f_b=ω_b/2pi
f_p=abs(ω_p/2pi)
plot(ωk_chain/2pi,Energies_normal_modes_chain)
vline!([f_a],linestyle=:dash, color=:red)
annotate!((f_a)-.2,1e-0, Plots.text(L"\omega_d/2+\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-2f_p],linestyle=:dash, color=:red)
annotate!((f_a-2f_p)-.2,1e-0, Plots.text(L"\omega_d/2-3\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+2f_p],linestyle=:dash, color=:red)
annotate!((f_a+2f_p)-.2,1e-0, Plots.text(L"\omega_d/2+5\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-4f_p],linestyle=:dash, color=:red)
annotate!((f_a-4f_p)-.2,1e-0, Plots.text(L"\omega_d/2-7\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+4f_p],linestyle=:dash, color=:red)
annotate!((f_a+4f_p)-.2,1e-0, Plots.text(L"\omega_d/2+9\omega_p/2",18, :red, rotation = 90 ))

vline!([f_p],linestyle=:dash, color=:red)
annotate!(f_p-.2,1e-0, Plots.text(L"\omega_p", 18, :red, rotation = 90 ))
vline!([3f_p],linestyle=:dash, color=:red)
annotate!(3f_p-.2,1e-0, Plots.text(L"3\omega_p", 18, :red, rotation = 90 ))

vline!([f_b],linestyle=:dash, color=:red)
annotate!(f_b-.2,1e-8, Plots.text(L"\omega_d",18, :red, rotation = 90 ))
vline!([f_b-2f_p],linestyle=:dash, color=:red)
annotate!((f_b-2f_p)-.2,1e-8, Plots.text(L"\omega_d-2\omega_p",18, :red, rotation = 90 ))
vline!([f_b+2f_p],linestyle=:dash, color=:red)
annotate!((f_b+2f_p)-.2,1e-8, Plots.text(L"\omega_d+2\omega_p",18, :red, rotation = 90 ))
#vline!([f_b-4f_p],linestyle=:dash, color=:red)
#annotate!((f_b-4f_p)-.2,1e-8, Plots.text(L"\omega_d-4\omega_p",18, :red, rotation = 90 ))
vline!([f_b+4f_p],linestyle=:dash, color=:red)
annotate!((f_b+4f_p)-.2,1e-8, Plots.text(L"\omega_d+4\omega_p",18, :red, rotation = 90 ))

vline!([3f_b/2-5f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2-5f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-5\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2-f_p/2],linestyle=:dash, color=:red) 
annotate!((3f_b/2-f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+3f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+3f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+3\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+7f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+7f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+7\omega_p/2",18, :red, rotation = 90 ))

#vline!([2f_b-3f_p],linestyle=:dash, color=:red)
#annotate!((2f_b-3f_p)-.2,1e-8, Plots.text(L"2\omega_d-3\omega_p",18, :red, rotation = 90 ))
vline!([2f_b-f_p],linestyle=:dash, color=:red)
annotate!((2f_b-f_p)-.2,1e-8, Plots.text(L"2\omega_d-\omega_p",18, :red, rotation = 90 ))
vline!([2f_b+f_p],linestyle=:dash, color=:red)
annotate!((2f_b+f_p)-.2,1e-8, Plots.text(L"2\omega_d+\omega_p",18, :red, rotation = 90 ))

plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 16), ylims=(1e-10, 1e2))
#savefig("../../Figures/Emission_spectrum/Energies_normal_modes_chain_two_photon_09_07.pdf")

In [ ]:
peaks_freq = [f_p, 3*f_p, f_a-4f_p, f_a-2f_p, f_a, f_a+2f_p, f_a+4f_p, f_b-2f_p, f_b, f_b+2f_p, f_b+4f_p, 3f_b/2-5f_p/2, 3f_b/2-f_p/2, 3f_b/2+3f_p/2, 3f_b/2+7f_p/2, 2f_b-f_p, 2f_b+f_p];

In [ ]:
ϵ_0_init = 0.3
ϵ_0_f = 0.9
ϵ_0_list= ϵ_0_init:0.01:ϵ_0_f

Energys_vs_ϵ_0_list = zeros(length(ϵ_0_list), length(ωk_chain))

for (i,ϵ_0_i) in ProgressBar(enumerate(ϵ_0_list))
    #g2_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b *1e3/2pi# (Leon's paper)
    ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0_i,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=true)

    # Solve P_chain * S = ϕ for S instead of inverting
    S = P_chain \ ϕ
    S_dot = P_chain \ ϕ_dot

    # Energy in normal modes
    Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2
    Energys_vs_ϵ_0_list[i,:] = Energies_normal_modes_chain
end



In [ ]:
g2_list =@. besselj(1,ϵ_0_list)/besselj(1,0.6*pi)
heatmap(ωk_chain/2pi,g2_list, log10.(Energys_vs_ϵ_0_list))
for peaks in peaks_freq
    #vline!([peaks],linestyle=:dash, color=:red, label="", alpha=0.5, linewidth=1)
end
plot!(xlabel=L"\omega_k/2\pi"*" (GHZ)", ylabel=L"g_2/g_{2,max}",
 title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)",
 colorbar_title = "\n"*L"\log_{10}"*"(Energy)",
 xlims=(.05, 15.5),
 right_margin = 20mm)

#savefig("../../Figures/Emission_spectrum/Energies_vs_g2_heatmap.pdf")

In [ ]:
g2_typical = 30*1e-3*2pi
ϵ_d_list = (0.5:0.5:12)*g2_typical

In [ ]:
Energys_vs_ϵ_d_list = zeros(length(ϵ_d_list), length(ωk_chain))

for (i,ϵ_d_i) in ProgressBar(enumerate(ϵ_d_list))
    #g2_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b *1e3/2pi# (Leon's paper)
    ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0, κ=κ, ϵ_d=ϵ_d_i, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0), β=∞,minimal_a=false,emission=true)

    # Solve P_chain * S = ϕ for S instead of inverting
    S = P_chain \ ϕ
    S_dot = P_chain \ ϕ_dot

    # Energy in normal modes
    Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2
    Energys_vs_ϵ_d_list[i,:] = Energies_normal_modes_chain
end

In [ ]:
heatmap(ωk_chain/2pi,ϵ_d_list, log10.(Energys_vs_ϵ_d_list))

plot!(xlabel=L"\omega_k/2\pi"*" (GHZ)", ylabel=L"\varepsilon_d",
 title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)",
 colorbar_title = "\n"*L"\log_{10}"*"(Energy)",
 xlims=(.05, 15.5),
 right_margin = 20mm)

#savefig("../../Figures/Emission_spectrum/Energies_vs_ϵ_d_heatmap.pdf")

In [ ]:
Energy_max_eps_0 = zeros(length(peaks_freq), length(ϵ_0_list))
Energy_max_eps_d = zeros(length(peaks_freq), length(ϵ_d_list))

for (i, peaks) in enumerate(peaks_freq)
    freqs_close = argmin(abs.(ωk_chain[:]/2pi .- peaks))
    # Take the maximum along the 2nd dimension (columns) for each row (for each ϵ_0)
    Energy_max_eps_0[i, :] = maximum(Energys_vs_ϵ_0_list[:, max(freqs_close-20,1):min(freqs_close+100, size(Energys_vs_ϵ_0_list,2))], dims=2)[:]
    Energy_max_eps_d[i, :] = maximum(Energys_vs_ϵ_d_list[:, max(freqs_close-20,1):min(freqs_close+100, size(Energys_vs_ϵ_d_list,2))], dims=2)[:]
end

In [ ]:
g2_list =@. besselj(1,ϵ_0_list)/besselj(1,0.6*pi)
p0=plot()
for (i,peaks) in enumerate(peaks_freq)
    peaks_round=round(peaks, digits=2)
    color_index = (i-1)*div(length(Perso_colormap)-1,length(peaks_freq)-1)+1
    plot!(g2_list, Energy_max_eps_0[i,:]/Energy_max_eps_0[i,1], label="$(peaks_round) GHz", color=Perso_colormap[color_index])
end
p1 = plot(p0,xlabel=L"g_2/g_{2,max}", ylabel="Energy in normal modes (normalized)",
 legend=:topleft,legend_columns=2)
p2 = plot(p0,xlabel=L"g_2/g_{2,max}", ylabel="Energy in normal modes (normalized)", 
legend=:bottomright,legend_columns=2, 
yaxis=:log,
xaxis=:log,
ylims=(1e-5, 1e2))
plot_perso(p1, p2, layout=(1,2), title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)")
#savefig("../../Figures/Emission_spectrum/Energies_vs_g2_cut.pdf")

In [ ]:
p0=plot()
for (i,peaks) in enumerate(peaks_freq)
    peaks_round=round(peaks, digits=2)
    color_index = (i-1)*div(length(Perso_colormap)-1,length(peaks_freq)-1)+1
    plot!(ϵ_d_list, Energy_max_eps_d[i,:]/Energy_max_eps_d[i,1], label="$(peaks_round) GHz", color=Perso_colormap[color_index])
end
p1 = plot(p0,xlabel=L"\varepsilon_d", ylabel="Energy in normal modes (normalized)",
 legend=:topleft,legend_columns=2)
p2 = plot(p0,xlabel=L"\varepsilon_d", ylabel="Energy in normal modes (normalized)", 
legend=:bottomright,legend_columns=2, 
yaxis=:log,
xaxis=:log,
ylims=(1e-5, 1e5))
plot_perso(p1, p2, layout=(1,2), title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)")
#savefig("../../Figures/Emission_spectrum/Energies_vs_ϵ_d_cut.pdf")

In [ ]:
p0=plot()
for (i,peaks) in enumerate(peaks_freq)
    peaks_round=round(peaks, digits=2)
    color_index = (i-1)*div(length(Perso_colormap)-1,length(peaks_freq)-1)+1
    plot!(g2_list, Energy_max_eps_0[i,:], label="$(peaks_round) GHz", color=Perso_colormap[color_index])
end
p1 = plot(p0,xlabel=L"\varepsilon_d", ylabel="Energy in normal modes (normalized)",
 legend=:topleft,legend_columns=2)
p2 = plot(p0,xlabel=L"\varepsilon_d", ylabel="Energy in normal modes (normalized)", 
legend=false,legend_columns=2, 
yaxis=:log,
xaxis=:log,
ylims=(1e-10, 1e5))
plot_perso(p1, p2, layout=(1,2), title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)")
#savefig("../../Figures/Emission_spectrum/Energies_vs_ϵ_d_cut.pdf")

# We now look at the third order expansion

In [ ]:
function two_photon_1d_chain_3rd_order(n, t_max; chain_params=nothing,
     circuit_params=(ω_a=4*2*pi,ω_b=7.05*2*pi,E_J = 37*2*pi, E_Ca=0.0121, E_Cb=0.0705, ϵ_0 = 0.6*pi,κ=1/10,ϵ_d=0.24,ω_d=7.0*2*pi),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0),cat_phase=0.0,minimal_a=false,emission=false, fast_no_chain=true)



    # Intialization of the circuit parameters
    ω_a, ω_b, E_J, E_Ca, E_Cb, ϵ_0, κ, ϵ_d, ω_d = circuit_params
    C_a_inv=2*E_Ca # hbar/e^2 GHz
    C_b_inv=2*E_Cb # hbar/e^2 GHz
    Z_a=C_a_inv/ω_a # Impedance of the resonator a
    Z_b=C_b_inv/ω_b # Impedance of the resonator b
    

    ϕ_quantum=1/2 #flux quantum in hbar/e


    # Initialization of the bath parameters
    onsite_energies, hopping_energies, system_chain_coupling = chain_params
    

    function drive(t)
        return ϵ_d * cos(ω_d * t + cat_phase) #We consider a drive resonant with the mode b
    end

    ω_p=2ω_a-ω_d
    function epsilon(t)
        return ϵ_0 * cos(ω_p * t) # We consider a drive resonant with the two-photon process
    end


    temp_buf = zeros(n)
    function two_photon_ODE(ddu,du,u,p,t)
        ϕ_a = u[1]
        ϕ_b = u[2]
        Φ=u[3:end]

        
        ddu[1] = -ω_a^2*ϕ_a + 2*E_J*ω_a * sqrt(Z_a) * sin(epsilon(t))*((ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum)^2)/2 / ϕ_quantum
        ddu[2] = -ω_b^2*ϕ_b + ω_b * system_chain_coupling * sqrt(Z_b) * Φ[1] - ω_b *drive(t)*sqrt(2) + 2*E_J*ω_b * sqrt(Z_b) * sin(epsilon(t))*((ϕ_a * sqrt(Z_a)/ϕ_quantum + ϕ_b * sqrt(Z_b)/ϕ_quantum)^2)/2 / ϕ_quantum
        mul!(temp_buf, Mϕ_chain, Φ)
        @. temp_buf = -temp_buf
        temp_buf[1] += system_chain_coupling * sqrt(Z_b) * ϕ_b
        mul!(view(ddu, 3:length(ddu)), Mq_chain, temp_buf)
        return nothing
    end


    gq=vcat(- hopping_energies)
    gϕ=vcat(hopping_energies)

    Mq_chain=SymTridiagonal(onsite_energies, gq)
    Mϕ_chain=SymTridiagonal(onsite_energies, gϕ)

    Mq_full=SymTridiagonal(vcat([ω_a,ω_b], onsite_energies), vcat([0,0],gq))

    #Initial conditions
    ϕ_a_0,q_a_0 = init_mode_a

    ϕ_b_0,q_b_0 = init_mode_b


    ϕ_0 = vcat(ϕ_a_0/sqrt(Z_a),ϕ_b_0/sqrt(Z_b), fill(0.0, n))
    q_0 = vcat(q_a_0 * sqrt(Z_a),q_b_0 * sqrt(Z_b), fill(0.0, n))
    ϕ_dot_0 = Mq_full*q_0


    #Run the simulation
    tspan = (0.0, t_max) # Time span for the simulation
    dt = 1 / (5 * maximum(vcat(ω_p,ω_a,ω_b,onsite_energies))) # Time step for the simulation
    prob = SecondOrderODEProblem(two_photon_ODE, ϕ_dot_0, ϕ_0, tspan)

    if emission
        save_idxs=vcat(3:n+2,n+5:n+4+n) # We save the chain DoFs (Φ)
        sol = solve(prob, VelocityVerlet(),dt=dt,
        saveat=t_max,                      # how often to save (or choose a coarser step)
        save_idxs=save_idxs,            # only store system DoFs
        save_everystep=false            # do not store intermediate solver steps
        )
        ϕ_dot = [getindex.(sol.u,i)[end] for i in 1:n]
        ϕ = [getindex.(sol.u,i+n)[end] for i in 1:n]
        return(ϕ,ϕ_dot,sol.t,Mq_chain, Mϕ_chain)
    
    end
    
    if fast_no_chain
        save_idxs=[1,2,n+3,n+4] # We only save the system DoFs (ϕ_a, ϕ_b) and not the chain DoFs (Φ)
        sol = solve(prob, VelocityVerlet(),dt=dt,
        saveat=dt,                      # how often to save (or choose a coarser step)
        save_idxs=save_idxs,            # only store system DoFs
        save_everystep=false            # do not store intermediate solver steps
        )

        ϕ_dot = [getindex.(sol.u,i) for i in 1:2]
        ϕ = [getindex.(sol.u,i+2) for i in 1:2]
    
        ϕ_a = ϕ[1] * sqrt(Z_a)
        q_a = ϕ_dot[1]/ω_a / sqrt(Z_a)
        if minimal_a
            return(ϕ_a,q_a,sol.t)
        end

        ϕ_b = ϕ[2] * sqrt(Z_b)
        q_b = ϕ_dot[2]/ω_b / sqrt(Z_b)

        t_list = sol.t
        return(ϕ_a,ϕ_b,q_a,q_b,t_list)
    end

    save_idxs=vcat(3:n+2,n+5:n+4+n) # We save the chain DoFs (Φ)
    saveat= t_max # We save every 100 steps
    n_red = length(save_idxs) # Number of reduced DoFs to save
    sol = solve(prob, VelocityVerlet(),dt=dt,
    saveat=saveat,                      # how often to save (or choose a coarser step)
    save_idxs=save_idxs,            # only store system DoFs
    save_everystep=false            # do not store intermediate solver steps
    )
    ϕ_dot = [getindex.(sol.u,i)[end] for i in 1:n]
    ϕ = [getindex.(sol.u,n+i)[end] for i in 1:n]
    return(ϕ,ϕ_dot,saveat)
end

In [ ]:
ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain_3rd_order(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0),minimal_a=false,emission=true)

In [ ]:
# Solve P_chain * S = ϕ for S instead of inverting
S = P_chain \ ϕ
S_dot = P_chain \ ϕ_dot

# Energy in normal modes
Energies_normal_modes_chain_3 = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2;

In [ ]:
f_a=ω_a/2pi
f_b=ω_b/2pi
f_p=abs(ω_p/2pi)
plot(ωk_chain/2pi,Energies_normal_modes_chain_3)
vline!([f_a],linestyle=:dash, color=:red)
annotate!((f_a)-.2,1e-0, Plots.text(L"\omega_d/2+\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-2f_p],linestyle=:dash, color=:red)
annotate!((f_a-2f_p)-.2,1e-0, Plots.text(L"\omega_d/2-3\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+2f_p],linestyle=:dash, color=:red)
annotate!((f_a+2f_p)-.2,1e-0, Plots.text(L"\omega_d/2+5\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a-4f_p],linestyle=:dash, color=:red)
annotate!((f_a-4f_p)-.2,1e-0, Plots.text(L"\omega_d/2-7\omega_p/2",18, :red, rotation = 90 ))
vline!([f_a+4f_p],linestyle=:dash, color=:red)
annotate!((f_a+4f_p)-.2,1e-0, Plots.text(L"\omega_d/2+9\omega_p/2",18, :red, rotation = 90 ))

vline!([f_p],linestyle=:dash, color=:red)
annotate!(f_p-.2,1e-0, Plots.text(L"\omega_p", 18, :red, rotation = 90 ))
vline!([3f_p],linestyle=:dash, color=:red)
annotate!(3f_p-.2,1e-0, Plots.text(L"3\omega_p", 18, :red, rotation = 90 ))

vline!([f_b],linestyle=:dash, color=:red)
annotate!(f_b-.2,1e-8, Plots.text(L"\omega_d",18, :red, rotation = 90 ))
vline!([f_b-2f_p],linestyle=:dash, color=:red)
annotate!((f_b-2f_p)-.2,1e-8, Plots.text(L"\omega_d-2\omega_p",18, :red, rotation = 90 ))
vline!([f_b+2f_p],linestyle=:dash, color=:red)
annotate!((f_b+2f_p)-.2,1e-8, Plots.text(L"\omega_d+2\omega_p",18, :red, rotation = 90 ))
#vline!([f_b-4f_p],linestyle=:dash, color=:red)
#annotate!((f_b-4f_p)-.2,1e-8, Plots.text(L"\omega_d-4\omega_p",18, :red, rotation = 90 ))
vline!([f_b+4f_p],linestyle=:dash, color=:red)
annotate!((f_b+4f_p)-.2,1e-8, Plots.text(L"\omega_d+4\omega_p",18, :red, rotation = 90 ))

vline!([3f_b/2-5f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2-5f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-5\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2-f_p/2],linestyle=:dash, color=:red) 
annotate!((3f_b/2-f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2-\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+3f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+3f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+3\omega_p/2",18, :red, rotation = 90 ))
vline!([3f_b/2+7f_p/2],linestyle=:dash, color=:red)
annotate!((3f_b/2+7f_p/2)-.2,1e-8, Plots.text(L"3\omega_d/2+7\omega_p/2",18, :red, rotation = 90 ))

#vline!([2f_b-3f_p],linestyle=:dash, color=:red)
#annotate!((2f_b-3f_p)-.2,1e-8, Plots.text(L"2\omega_d-3\omega_p",18, :red, rotation = 90 ))
vline!([2f_b-f_p],linestyle=:dash, color=:red)
annotate!((2f_b-f_p)-.2,1e-8, Plots.text(L"2\omega_d-\omega_p",18, :red, rotation = 90 ))
vline!([2f_b+f_p],linestyle=:dash, color=:red)
annotate!((2f_b+f_p)-.2,1e-8, Plots.text(L"2\omega_d+\omega_p",18, :red, rotation = 90 ))

plot!(ylabel="Energy stored in the chain", xlabel="Frequency (GHz)", title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)", legend=false, yaxis=:log, xlims=(0.1, 16), ylims=(1e-10, 1e2))
#savefig("../../Figures/Emission_spectrum/Energies_normal_modes_chain_two_photon_09_07.pdf")

In [ ]:
ϵ_0_init = 0.3
ϵ_0_f = 0.9
ϵ_0_list= ϵ_0_init:0.01:ϵ_0_f

Energys_vs_ϵ_0_list_3 = zeros(length(ϵ_0_list), length(ωk_chain))

for (i,ϵ_0_i) in ProgressBar(enumerate(ϵ_0_list))
    #g2_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b *1e3/2pi# (Leon's paper)
    ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain_3rd_order(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0_i,κ=κ, ϵ_d=ϵ_d, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0),minimal_a=false,emission=true)

    # Solve P_chain * S = ϕ for S instead of inverting
    S = P_chain \ ϕ
    S_dot = P_chain \ ϕ_dot

    # Energy in normal modes
    Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2
    Energys_vs_ϵ_0_list_3[i,:] = Energies_normal_modes_chain
end



In [ ]:
g2_list =@. besselj(1,ϵ_0_list)/besselj(1,0.6*pi)
heatmap(ωk_chain/2pi,g2_list, log10.(Energys_vs_ϵ_0_list_3))
for peaks in peaks_freq
    #vline!([peaks],linestyle=:dash, color=:red, label="", alpha=0.5, linewidth=1)
end
plot!(xlabel=L"\omega_k/2\pi"*" (GHZ)", ylabel=L"g_2/g_{2,max}",
 title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)",
 colorbar_title = "\n"*L"\log_{10}"*"(Energy)",
 xlims=(.05, 15.5),
 right_margin = 20mm)

#savefig("../../Figures/Emission_spectrum/Energies_vs_g2_heatmap_3.pdf")

In [ ]:
g2_typical = 30*1e-3*2pi
ϵ_d_list = (1:.5:12)*g2_typical

In [ ]:
Energys_vs_ϵ_d_list_3 = zeros(length(ϵ_d_list), length(ωk_chain))

for (i,ϵ_d_i) in ProgressBar(enumerate(ϵ_d_list))
    #g2_i = besselj(1,ϵ_0_i)*E_J*φ_a^2*φ_b *1e3/2pi# (Leon's paper)
    ϕ,ϕ_dot,t_list,_,_ = two_photon_1d_chain_3rd_order(n, t_max; chain_params=chain_params,
     circuit_params=(ω_a=ω_a,ω_b=ω_b,E_J =E_J, E_Ca=E_Ca, E_Cb=E_Cb, ϵ_0 = ϵ_0, κ=κ, ϵ_d=ϵ_d_i, ω_d=ω_b_0),
     init_mode_a=(ϕ_a_0=0.0,q_a_0 = 0.0), init_mode_b=(ϕ_b_0=0.0,q_b_0 = 0.0),minimal_a=false,emission=true)

    # Solve P_chain * S = ϕ for S instead of inverting
    S = P_chain \ ϕ
    S_dot = P_chain \ ϕ_dot

    # Energy in normal modes
    Energies_normal_modes_chain = Dq_chain .* S_dot.^2 .+ Dϕ_chain .* S.^2
    Energys_vs_ϵ_d_list_3[i,:] = Energies_normal_modes_chain
end

In [ ]:
heatmap(ωk_chain/2pi,ϵ_d_list, log10.(Energys_vs_ϵ_d_list_3))

plot!(xlabel=L"\omega_k/2\pi"*" (GHZ)", ylabel=L"\varepsilon_d",
 title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)",
 colorbar_title = "\n"*L"\log_{10}"*"(Energy)",
 xlims=(.05, 15.5),
 right_margin = 20mm)

#savefig("../../Figures/Emission_spectrum/Energies_vs_ϵ_d_heatmap_3.pdf")

In [ ]:
Energy_max_eps_0_3 = zeros(length(peaks_freq), length(ϵ_0_list))
Energy_max_eps_d_3 = zeros(length(peaks_freq), length(ϵ_d_list))

for (i, peaks) in enumerate(peaks_freq)
    freqs_close = argmin(abs.(ωk_chain[:]/2pi .- peaks))
    # Take the maximum along the 2nd dimension (columns) for each row (for each ϵ_0)
    Energy_max_eps_0_3[i, :] = maximum(Energys_vs_ϵ_0_list_3[:, max(freqs_close-20,1):min(freqs_close+100, size(Energys_vs_ϵ_0_list_3,2))], dims=2)[:]
    Energy_max_eps_d_3[i, :] = maximum(Energys_vs_ϵ_d_list_3[:, max(freqs_close-20,1):min(freqs_close+100, size(Energys_vs_ϵ_d_list_3,2))], dims=2)[:]
end

In [ ]:
g2_list =@. besselj(1,ϵ_0_list)/besselj(1,0.6*pi)
p0=plot()
for (i,peaks) in enumerate(peaks_freq)
    peaks_round=round(peaks, digits=2)
    color_index = (i-1)*div(length(Perso_colormap)-1,length(peaks_freq)-1)+1
    plot!(g2_list, Energy_max_eps_0_3[i,:]/Energy_max_eps_0_3[i,1], label="$(peaks_round) GHz", color=Perso_colormap[color_index])
end
p1 = plot(p0,xlabel=L"g_2/g_{2,max}", ylabel="Energy in normal modes (normalized)",
 legend=:topleft,legend_columns=2)
p2 = plot(p0,xlabel=L"g_2/g_{2,max}", ylabel="Energy in normal modes (normalized)", 
legend=:bottomright,legend_columns=2, 
yaxis=:log,
xaxis=:log,
ylims=(1e-5, 1e2))
plot_perso(p1, p2, layout=(1,2), title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)")
#savefig("../../Figures/Emission_spectrum/Energies_vs_g2_cut_3.pdf")

In [ ]:
p0=plot()
for (i,peaks) in enumerate(peaks_freq)
    peaks_round=round(peaks, digits=2)
    color_index = (i-1)*div(length(Perso_colormap)-1,length(peaks_freq)-1)+1
    plot!(ϵ_d_list, Energy_max_eps_d_3[i,:]/Energy_max_eps_d_3[i,1], label="$(peaks_round) GHz", color=Perso_colormap[color_index])
end
p1 = plot(p0,xlabel=L"\varepsilon_d", ylabel="Energy in normal modes (normalized)",
 legend=:topleft,legend_columns=2)
p2 = plot(p0,xlabel=L"\varepsilon_d", ylabel="Energy in normal modes (normalized)", 
legend=:bottomright,legend_columns=2, 
yaxis=:log,
xaxis=:log,
ylims=(1e-5, 1e5))
plot_perso(p1, p2, layout=(1,2), title = L"κt = "*"$(κ*t_max), "*L"n_{osc}="*"$(n)")
#savefig("../../Figures/Emission_spectrum/Energies_vs_ϵ_d_cut_3.pdf")